# Backtest Markowitz — Tres estrategias de cartera semanal

Este notebook compara tres formas de estimar la rentabilidad esperada dentro de un marco de optimización de Markowitz:

| ID | Estrategia | Rentabilidad esperada usada |
|----|------------|-----------------------------|
| BT_001 | Markowitz_media_historica | Media histórica de retornos hasta la semana anterior |
| BT_002 | Markowitz_ultimo_valor | Retorno semanal observado la semana anterior |
| BT_003 | Markowitz_ARIMA | Predicción ARIMA walk-forward |

**Período de backtest**: 2024-W30 → 2026-W05  
**Período de entrenamiento inicial**: 2021-W01 → 2024-W29  
**Capital inicial**: 100  
**Sin posiciones cortas | Pesos suman 1 | Peso máximo por ETF: 40%**

In [15]:
# pip install PyPortfolioOpt cvxpy osqp gspread google-auth

In [16]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp
from datetime import datetime
from pypfopt import risk_models, EfficientFrontier

import gspread
from google.oauth2.service_account import Credentials

warnings.filterwarnings('ignore')


## Configuración

In [17]:
DATA_DIR = '../Datos_csv'
SERVICE_ACCOUNT_FILE = '../3.Modelo_ML/Modelos_semanales/credenciales_google.json'
SHEET_URL = 'https://docs.google.com/spreadsheets/d/1_4qMy2c3DS52F9adfARbPkKVLyxeK1j2GVmf4ZOLy1Q/edit'
SCOPES = [
    'https://www.googleapis.com/auth/spreadsheets',
    'https://www.googleapis.com/auth/drive',
]

# Split temporal (mismo que modelos ML)
FECHA_TRAIN_FIN = '2024-W29'
FECHA_TEST_INI  = '2024-W30'
FECHA_TEST_FIN  = '2026-W05'

# Parámetros de cartera
CAPITAL_INICIAL = 100.0
MAX_WEIGHT      = 0.40   # peso máximo por ETF (0 a 1); None = sin límite
RISK_AVERSION   = 10.0   # lambda: mayor -> más conservador

print('Configuración cargada')

Configuración cargada


## Carga de datos

In [18]:
# Precios ajustados
adj_close = pd.read_csv(
    os.path.join(DATA_DIR, 'adj_close.csv'),
    index_col=0, parse_dates=True
)

# ETFs válidos (pasan filtro de estacionalidad)
with open(os.path.join(DATA_DIR, 'tickers_estacionalidad.json'), 'r', encoding='utf-8') as f:
    est_data = json.load(f)
tickers_filtrados = est_data['tickers_filtrados']

# Retornos logarítmicos diarios solo para ETFs válidos
ret_log_diff = np.log(adj_close[tickers_filtrados]).diff().iloc[1:]

# Clave semana ISO (YYYY-Wnn)
iso_cal  = ret_log_diff.index.isocalendar()
week_key = (
    iso_cal.year.astype(str) + '-W' + iso_cal.week.astype(str).str.zfill(2)
).values

# Retorno semanal = media de retornos diarios de la semana
ret_weekly_df = ret_log_diff.groupby(week_key).mean()
ret_weekly_df.index.name = 'week_key'

print(f'ETFs validos: {len(tickers_filtrados)}')
print(f'Semanas totales: {len(ret_weekly_df)} | {ret_weekly_df.index[0]} -> {ret_weekly_df.index[-1]}')
ret_weekly_df.head(3)

ETFs validos: 58
Semanas totales: 266 | 2021-W01 -> 2026-W06


,AGG,BND,DBC,DIA,DVY,EEM,EFA,EWG,EWJ,EWQ,...,XLB,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
week_key,,,,,,,,,,,,,,,,,,,,,
2021-W01,-0.002042,-0.002281,0.010383,0.007109,0.012052,0.012701,0.008017,0.005839,0.009299,0.006168,...,0.016233,0.021808,0.015338,0.009091,0.005529,0.000674,0.001971,0.004901,0.009639,0.014278
2021-W02,0.000290,0.000183,0.000786,-0.001804,0.002915,-0.001320,-0.003544,-0.006671,-0.001652,-0.005888,...,-0.003082,0.006321,0.000129,-0.001748,-0.005175,-0.003833,0.003778,0.002107,-0.000717,-0.003575
2021-W03,0.000000,0.000086,-0.003455,0.001497,-0.002190,0.006717,0.002508,0.005514,0.001885,0.000075,...,-0.003206,-0.004009,-0.004896,-0.000902,0.010395,-0.002142,0.003281,-0.000596,0.001322,0.006401


In [19]:
# Carga de predicciones ARIMA generadas por el notebook de series temporales semanal
# Ejecutar primero: 2.Modelo_Series_Temporales/Modelo_semanal/01_datos_series_temporales_grupal_semanal.ipynb
# (celdas del bloque 'Predicciones ARIMA para Markowitz')

arima_path = os.path.join(DATA_DIR, 'predicciones_arima_semanal.csv')
df_arima = pd.read_csv(arima_path, encoding='utf-8-sig')

# Pivote: semana x ETF -> predicción ARIMA
pred_arima_pivot = df_arima.pivot(
    index='Fecha_Semana', columns='Target', values='Prediccion'
)
pred_arima_pivot.index.name = 'week_key'

print(f'Predicciones ARIMA: {pred_arima_pivot.shape}')
print(f'Rango: {pred_arima_pivot.index[0]} -> {pred_arima_pivot.index[-1]}')
pred_arima_pivot.head(3)

Predicciones ARIMA: (81, 58)
Rango: 2024-W30 -> 2026-W06


Target,AGG,BND,DBC,DIA,DVY,EEM,EFA,EWG,EWJ,EWQ,...,XLB,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
week_key,,,,,,,,,,,,,,,,,,,,,
2024-W30,0.000258,0.000256,0.000407,-0.000055,-0.000025,0.000522,3.500000e-06,-0.000438,0.000198,0.000064,...,0.000030,0.000061,-0.000438,-0.000218,0.000848,-0.000137,0.000571,-0.000288,0.000012,0.000398
2024-W31,-0.000106,-0.000114,0.000204,-0.000054,-0.000024,0.000144,9.000000e-08,0.000176,0.000185,-0.000005,...,-0.000096,-0.001006,-0.000177,0.000145,0.000282,-0.000084,-0.001246,0.000238,-0.000085,0.000561
2024-W32,0.000157,0.000149,0.000190,0.000162,0.000025,0.000227,-1.020000e-06,-0.000448,0.000430,0.000083,...,0.000105,0.000548,-0.000597,0.001069,0.000670,-0.000160,0.000762,0.000653,-0.000037,0.000682


## Optimizador Markowitz (scipy)

Maximiza la utilidad cuadrática:
$$\max_w \; w^\top \mu - \frac{\lambda}{2} w^\top \Sigma w$$
Sujeto a: $\sum w_i = 1$, $0 \leq w_i \leq w_{\max}$

La covarianza se estima con el método **Ledoit-Wolf** (shrinkage), que es más robusto con pocas observaciones.

In [20]:
def calcular_covarianza_lw(ret_hist: pd.DataFrame) -> pd.DataFrame:
    """Matriz de covarianza Ledoit-Wolf annualizada (x52 semanas)."""
    Sigma = risk_models.CovarianceShrinkage(
        ret_hist, returns_data=True, frequency=52 
    ).ledoit_wolf()
    return risk_models.fix_nonpositive_semidefinite(Sigma) #matriz definida positiva, frecuencia: hay 52 semanas en un año


def optimizar_markowitz(
    mu: np.ndarray,
    cov: pd.DataFrame,
    etfs: list,
    max_weight: float = 0.30,
    risk_aversion: float = 10.0
) -> dict:
    """
    Optimiza la cartera con PyPortfolioOpt (EfficientFrontier).
    Devuelve dict {etf: peso}. Si falla, equal-weight.
    """
    n = len(etfs)
    mu_series = pd.Series(mu, index=etfs)

    solvers = set(cp.installed_solvers()) #motor matemático que resuelve la optimización.
    if "OSQP" in solvers:
        solver, solver_options = "OSQP", {"max_iter": 300000}
    elif "SCS" in solvers:
        solver, solver_options = "SCS", {"max_iters": 100000}
    else:
        raise RuntimeError(f"Sin solver. Instalados: {sorted(solvers)}")


    delta = 2 * risk_aversion

    try:
        ef = EfficientFrontier(
            mu_series, cov,
            weight_bounds=(0, max_weight),
            solver=solver,
            solver_options=solver_options
        )
        ef.max_quadratic_utility(risk_aversion=delta)
        return dict(ef.clean_weights())
    except Exception as e:
        print(f'  [FALLBACK equal-weight] {e}')
        return {etf: 1.0 / n for etf in etfs}


def retorno_cartera(pesos: dict, ret_real: pd.Series) -> float:
    """Retorno semanal de la cartera: sum(w_i * r_i)."""
    return sum(pesos.get(etf, 0.0) * ret_real.get(etf, 0.0) for etf in pesos)


## Backtest semanal

Para cada semana `t` del periodo de test:
1. **Covarianza**: se calcula con todos los retornos disponibles hasta `t` (incluida).
2. **Rentabilidad esperada**: según la estrategia.
   - *Media histórica*: media de todos los retornos hasta `t`.
   - *Último valor*: retorno observado en la semana `t` (último conocido).
   - *ARIMA*: predicción para la semana `t+1` hecha en `t`.
3. **Optimización**: cartera Markowitz con los inputs anteriores.
4. **Inversión**: se aplican los pesos a los retornos reales de `t+1`.
5. **Actualización**: capital acumulado.



In [ ]:
# Semanas del periodo de test
semanas_test = [
    s for s in ret_weekly_df.index
    if FECHA_TEST_INI <= s <= FECHA_TEST_FIN
]
print(f'Semanas de backtest: {len(semanas_test)} | {semanas_test[0]} -> {semanas_test[-1]}')

# ETFs con predicciones ARIMA y datos de retorno
etfs_comunes = [
    etf for etf in tickers_filtrados
    if etf in pred_arima_pivot.columns and etf in ret_weekly_df.columns
]
print(f'ETFs comunes: {len(etfs_comunes)}')

# Registros por estrategia
estrategia = {k: [] for k in ['media_historica', 'ultimo_valor', 'arima']}
capital   = {k: CAPITAL_INICIAL for k in estrategia}
capital_pico = {k: CAPITAL_INICIAL for k in estrategia} #maximo capitla q se ha tenido con esa estrategia

for i, semana_decision in enumerate(semanas_test[:-1]): #En semana_decision decides la cartera
    semana_inversion = semanas_test[i + 1] #En semana_inversion ves qué pasó realmente

    # Historial disponible hasta la semana de decisión (inclusive)
    hist = ret_weekly_df.loc[
        ret_weekly_df.index <= semana_decision, etfs_comunes
    ].dropna(how='all')

    # Covarianza Ledoit-Wolf (igual para las 3 estrategias)
    try:
        cov = calcular_covarianza_lw(hist[etfs_comunes].dropna())
    except Exception as e:
        print(f'[{semana_decision}] Error covarianza: {e}')
        continue

    # Retornos reales de la semana de inversión
    if semana_inversion not in ret_weekly_df.index:
        continue
    ret_real = ret_weekly_df.loc[semana_inversion, etfs_comunes]

    # Estrategia 1: Media histórica (anualizada) 
    mu_media = hist[etfs_comunes].mean().values * 52
    pesos_media = optimizar_markowitz(mu_media, cov, etfs_comunes, MAX_WEIGHT, RISK_AVERSION)

    # Estrategia 2: Último valor conocido (anualizado) 
    mu_ultimo = hist[etfs_comunes].iloc[-1].values * 52
    pesos_ultimo = optimizar_markowitz(mu_ultimo, cov, etfs_comunes, MAX_WEIGHT, RISK_AVERSION)

    # Estrategia 3: Predicción ARIMA
    # La predicción generada en semana_decision apunta a semana_inversion
    if semana_inversion in pred_arima_pivot.index:
        mu_arima_raw = pred_arima_pivot.loc[semana_inversion, etfs_comunes].fillna(0.0).values
    elif semana_decision in pred_arima_pivot.index:
        mu_arima_raw = pred_arima_pivot.loc[semana_decision, etfs_comunes].fillna(0.0).values
    else:
        mu_arima_raw = mu_media  # fallback a media
    mu_arima = mu_arima_raw * 52
    pesos_arima = optimizar_markowitz(mu_arima, cov, etfs_comunes, MAX_WEIGHT, RISK_AVERSION)

    # Calcular retornos y actualizar capital 
    estrategias = [
        ('media_historica', pesos_media,  mu_media),
        ('ultimo_valor',    pesos_ultimo, mu_ultimo),
        ('arima',           pesos_arima,  mu_arima),
    ]

    for nombre, pesos, mu_used in estrategias:
        ret_cart  = retorno_cartera(pesos, ret_real)
        cap_ini   = capital[nombre]
        cap_fin   = cap_ini * (1.0 + ret_cart)
        capital[nombre] = cap_fin

        capital_pico[nombre] = max(capital_pico[nombre], cap_fin)
        drawdown = (cap_fin - capital_pico[nombre]) / capital_pico[nombre]
        rent_acum = (cap_fin - CAPITAL_INICIAL) / CAPITAL_INICIAL

        row = {
            'FechaDecision':      semana_decision,
            'FechaInversion':     semana_inversion,
            'Estrategia':         nombre,
            'CapitalInicio':      round(cap_ini, 4),
            'RetornoCarteraReal': round(ret_cart, 6),
            'CapitalFinal':       round(cap_fin, 4),
            'RentabilidadAcumulada': round(rent_acum, 6),
            'Drawdown':           round(drawdown, 6),
            'SumaPesos':          round(sum(pesos.values()), 6),
            'Estado':             'DONE'
        }
        # Pesos por ETF
        for etf in etfs_comunes:
            row[f'{etf}_peso'] = round(pesos.get(etf, 0.0), 6)
        # Rentabilidad esperada usada (weekly, sin anualizar)
        for j, etf in enumerate(etfs_comunes):
            row[f'{etf}_rentexp'] = round(float(mu_used[j]) / 52, 6)
        # Retornos reales de esa semana
        for etf in etfs_comunes:
            val = ret_real.get(etf, np.nan)
            row[f'{etf}_real'] = round(float(val), 6) if not np.isnan(val) else None

        estrategia[nombre].append(row)

    if (i + 1) % 10 == 0:
        print(f'  Semana {i+1:3d}/{len(semanas_test)-1}: {semana_decision} -> {semana_inversion}')
        for k in capital:
            print(f'    {k}: {capital[k]:.2f}')

print('\n=== BACKTEST COMPLETADO ===')
for k, v in capital.items():
    rent = (v - CAPITAL_INICIAL) / CAPITAL_INICIAL * 100
    print(f'  {k}: Capital final = {v:.2f} | Rentabilidad = {rent:.2f}%')

Semanas de backtest: 80 | 2024-W30 -> 2026-W05
ETFs comunes: 58
  Semana  10/79: 2024-W39 -> 2024-W40
    media_historica: 99.57
    ultimo_valor: 102.60
    arima: 100.66
  Semana  20/79: 2024-W49 -> 2024-W50
    media_historica: 99.12
    ultimo_valor: 101.51
    arima: 100.38
  Semana  30/79: 2025-W07 -> 2025-W08
    media_historica: 99.06
    ultimo_valor: 103.86
    arima: 99.02
  Semana  40/79: 2025-W17 -> 2025-W18
    media_historica: 96.88
    ultimo_valor: 100.44
    arima: 98.96
  Semana  50/79: 2025-W27 -> 2025-W28
    media_historica: 98.52
    ultimo_valor: 99.78
    arima: 101.55
  Semana  60/79: 2025-W37 -> 2025-W38
    media_historica: 98.86
    ultimo_valor: 100.01
    arima: 102.74
  Semana  70/79: 2025-W47 -> 2025-W48
    media_historica: 99.56
    ultimo_valor: 100.80
    arima: 103.32

=== BACKTEST COMPLETADO ===
  media_historica: Capital final = 101.98 | Rentabilidad = 1.98%
  ultimo_valor: Capital final = 103.77 | Rentabilidad = 3.77%
  arima: Capital final = 10

In [ ]:
# Convertir a DataFrames
nombres_bt = {
    'media_historica': 'Markowitz_media_historica',
    'ultimo_valor':    'Markowitz_ultimo_valor',
    'arima':           'Markowitz_ARIMA'
}
ids_bt = {'media_historica': 'BT_001', 'ultimo_valor': 'BT_002', 'arima': 'BT_003'}

dfs = {k: pd.DataFrame(v) for k, v in registros.items()}
for nombre, df_e in dfs.items():
    print(f'{nombre}: {len(df_e)} semanas | Capital final: {df_e["CapitalFinal"].iloc[-1]:.4f}')

## Métricas de resumen

| Métrica | Descripción |
|---------|-------------|
| **CapitalFinal** | Valor de la cartera al final del backtest |
| **RentabilidadAcumulada** | (CapitalFinal − 100) / 100 |
| **Volatilidad** | Desviación estándar de retornos semanales × √52 (anualizada) |
| **Sharpe** | Retorno medio anualizado / Volatilidad |
| **MaxDrawdown** | Mayor caída desde un máximo previo hasta el siguiente mínimo |
| **SemanasPositivas** | Número de semanas con retorno positivo |

In [ ]:
resumen_rows = []
for nombre, df_e in dfs.items():
    if df_e.empty:
        continue
    rets  = df_e['RetornoCarteraReal'].values
    cap_f = df_e['CapitalFinal'].iloc[-1]
    rent_a = (cap_f - CAPITAL_INICIAL) / CAPITAL_INICIAL

    vol    = np.std(rets, ddof=1) * np.sqrt(52)
    sharpe = (np.mean(rets) * 52) / (vol + 1e-10)

    # Max Drawdown
    caps = df_e['CapitalFinal'].values
    peak, max_dd = caps[0], 0.0
    for c in caps:
        peak = max(peak, c)
        dd = (c - peak) / peak
        max_dd = min(max_dd, dd)

    resumen_rows.append({
        'ID':                   ids_bt[nombre],
        'Estrategia':           nombres_bt[nombre],
        'Frecuencia':           'Semanal',
        'FechaTrainIni':        '2021-W01',
        'FechaTrainFin':        FECHA_TRAIN_FIN,
        'FechaTestIni':         FECHA_TEST_INI,
        'FechaTestFin':         df_e['FechaInversion'].iloc[-1],
        'CapitalInicial':       CAPITAL_INICIAL,
        'CapitalFinal':         round(cap_f, 4),
        'RentabilidadAcumulada': f'{rent_a*100:.2f}%',
        'Volatilidad':          f'{vol*100:.2f}%',
        'Sharpe':               round(sharpe, 4),
        'MaxDrawdown':          f'{max_dd*100:.2f}%',
        'SemanasPositivas':     int((rets > 0).sum()),
        'Estado':               'DONE'
    })

df_resumen = pd.DataFrame(resumen_rows)
display(df_resumen)

## Evolución del capital

In [ ]:
colores = {
    'media_historica': '#4FC3F7',
    'ultimo_valor':    '#81C784',
    'arima':           '#EF739A'
}

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Panel 1: Capital acumulado
ax1 = axes[0]
for nombre, df_e in dfs.items():
    if not df_e.empty:
        ax1.plot(df_e['FechaInversion'], df_e['CapitalFinal'],
                 label=nombres_bt[nombre], color=colores[nombre], linewidth=1.8)
ax1.axhline(CAPITAL_INICIAL, color='gray', linewidth=0.8, linestyle='--', label='Capital inicial')
ax1.set_ylabel('Capital')
ax1.set_title('Evolución del capital — Backtest Markowitz semanal', fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)
plt.setp(ax1.get_xticklabels(), rotation=45, ha='right', fontsize=7)

# Panel 2: Retorno semanal
ax2 = axes[1]
for nombre, df_e in dfs.items():
    if not df_e.empty:
        ax2.plot(df_e['FechaInversion'], df_e['RetornoCarteraReal'],
                 label=nombres_bt[nombre], color=colores[nombre], linewidth=1.2, alpha=0.8)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_ylabel('Retorno semanal (log)')
ax2.set_title('Retorno semanal por estrategia', fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)
plt.setp(ax2.get_xticklabels(), rotation=45, ha='right', fontsize=7)

plt.tight_layout()
plt.savefig('backtest_markowitz.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado: 4.Markowitz/backtest_markowitz.png')

## Evolución de pesos (ejemplo: estrategia ARIMA)

In [ ]:
df_arima_bt = dfs['arima']
cols_peso = [c for c in df_arima_bt.columns if c.endswith('_peso')]
df_pesos = df_arima_bt[['FechaInversion'] + cols_peso].set_index('FechaInversion')
df_pesos.columns = [c.replace('_peso', '') for c in df_pesos.columns]

# Top ETFs por peso medio
top_etfs = df_pesos.mean().sort_values(ascending=False).head(10).index.tolist()

fig, ax = plt.subplots(figsize=(14, 5))
df_pesos[top_etfs].plot.area(ax=ax, alpha=0.8, colormap='tab10')
ax.set_title('Evolución de pesos — Markowitz ARIMA (top 10 ETFs)', fontweight='bold')
ax.set_ylabel('Peso')
ax.set_ylim(0, 1)
ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
ax.grid(alpha=0.2)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)
plt.tight_layout()
plt.show()

## Escritura en Google Sheets

**Hoja resumen_backtest** (gid=0): una fila por estrategia con métricas globales.  
**Hoja detalle_semanal** (gid=11355774): una fila por semana × estrategia con pesos, rentabilidades esperadas y retornos reales.

In [ ]:
def conectar_sheets():
    creds = Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)
    client = gspread.authorize(creds)
    return client.open_by_url(SHEET_URL)

try:
    spreadsheet = conectar_sheets()
    print('Google Sheets OK')
    print('Hojas:', [ws.title for ws in spreadsheet.worksheets()])
except Exception as e:
    print(f'Error de conexión: {e}')

In [ ]:
# ─── Hoja resumen_backtest (gid=0) ──────────────────────────────────────────
try:
    ws_resumen = spreadsheet.get_worksheet_by_id(0)
    ws_resumen.clear()

    headers = list(df_resumen.columns)
    ws_resumen.append_row(headers, value_input_option='RAW')

    for _, row in df_resumen.iterrows():
        ws_resumen.append_row(
            [v if isinstance(v, (int, float)) else str(v) for v in row.tolist()],
            value_input_option='RAW'
        )
    print(f'Hoja resumen_backtest: {len(df_resumen)} filas escritas')

except Exception as e:
    print(f'Error: {e}')
    import traceback; traceback.print_exc()

In [ ]:
# ─── Hoja detalle_semanal (gid=11355774) ────────────────────────────────────
try:
    ws_detalle = spreadsheet.get_worksheet_by_id(11355774)
    ws_detalle.clear()

    # Columnas fijas + columnas por ETF
    cols_base = [
        'ID_Backtest', 'FechaDecision', 'FechaInversion', 'Estrategia',
        'CapitalInicio', 'RetornoCarteraReal', 'CapitalFinal',
        'RentabilidadAcumulada', 'Drawdown', 'SumaPesos', 'Estado'
    ]
    cols_pesos   = [f'{e}_peso'    for e in etfs_comunes]
    cols_rentexp = [f'{e}_rentexp' for e in etfs_comunes]
    cols_real    = [f'{e}_real'    for e in etfs_comunes]
    all_cols = cols_base + cols_pesos + cols_rentexp + cols_real

    ws_detalle.append_row(all_cols, value_input_option='RAW')

    # Construir todas las filas
    rows_batch = []
    for nombre, df_e in dfs.items():
        for _, row in df_e.iterrows():
            fila = [ids_bt[nombre]]
            for col in cols_base[1:]:
                val = row.get(col, '')
                fila.append('' if (isinstance(val, float) and np.isnan(val)) else val)
            for col in cols_pesos + cols_rentexp + cols_real:
                val = row.get(col, '')
                if val is None or (isinstance(val, float) and np.isnan(val)):
                    fila.append('')
                else:
                    fila.append(val)
            rows_batch.append(fila)

    # Escribir en lotes de 500
    batch_size = 500
    for i in range(0, len(rows_batch), batch_size):
        ws_detalle.append_rows(
            rows_batch[i:i+batch_size],
            value_input_option='RAW'
        )
        print(f'  Escritas {min(i+batch_size, len(rows_batch))}/{len(rows_batch)} filas')

    print(f'\nHoja detalle_semanal: {len(rows_batch)} filas escritas')

except Exception as e:
    print(f'Error: {e}')
    import traceback; traceback.print_exc()

In [ ]:
print('=== RESUMEN FINAL ===')
print(f'Semanas de backtest: {len(semanas_test)-1}')
print(f'ETFs en cartera: {len(etfs_comunes)}')
print()
for nombre, df_e in dfs.items():
    if not df_e.empty:
        cap_f = df_e['CapitalFinal'].iloc[-1]
        rent  = (cap_f - CAPITAL_INICIAL) / CAPITAL_INICIAL * 100
        pos   = int((df_e['RetornoCarteraReal'] > 0).sum())
        print(f'  {nombres_bt[nombre]}')
        print(f'    Capital final = {cap_f:.2f} | Rentabilidad = {rent:.2f}% | Semanas+ = {pos}')

print()
print('Google Sheets:')
print('  Resumen: https://docs.google.com/spreadsheets/d/1_4qMy2c3DS52F9adfARbPkKVLyxeK1j2GVmf4ZOLy1Q/edit?gid=0')
print('  Detalle: https://docs.google.com/spreadsheets/d/1_4qMy2c3DS52F9adfARbPkKVLyxeK1j2GVmf4ZOLy1Q/edit?gid=11355774')